# Data Cleaning Pipeline

Reads `data/myeloma_data.csv`, selects relevant columns, renames them to `lower_snake_case`, maps ancestry, converts time from months to years, standardizes continuous features, and exports the result to `cleaned_data/`.

In [1]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
from pathlib import Path

DATA_PATH = 'data/myeloma_data.csv'
OUTPUT_DIR = Path('cleaned_data')
OUTPUT_DIR.mkdir(exist_ok=True)

## Load raw data

In [2]:
df_raw = pd.read_csv(DATA_PATH, index_col='ID')
print(f'Rows: {len(df_raw)}, Columns: {list(df_raw.columns)}')
df_raw.head()

Rows: 2000, Columns: ['Ancestry', 'PGS_Score', 'Age', 'M_Spike', 'sFLC_Ratio', 'Creatinine', 'LP', 'Time_Months', 'Status', 'BMPC_Final', 'Hgb_Final', 'PGS_Bin', 'Clinical_Risk']


,Ancestry,PGS_Score,Age,M_Spike,sFLC_Ratio,Creatinine,LP,Time_Months,Status,BMPC_Final,Hgb_Final,PGS_Bin,Clinical_Risk
ID,,,,,,,,,,,,,
1,European,-0.006056,69.111929,3.282018,3.010026,0.522791,2.172474,16.02,1,25.01001,14.598307,Medium PGS,Int Clinical Risk
2,European,0.996723,74.327334,3.468359,3.372553,1.028067,2.034360,0.80,1,NaN,15.478983,Medium PGS,Int Clinical Risk
3,European,0.723133,67.378696,0.588324,2.718493,0.775595,0.134618,107.38,0,0.15043,12.717573,Medium PGS,Low Clinical Risk
4,European,0.821519,72.769428,1.222824,6.164781,0.943936,2.541208,51.56,1,NaN,NaN,Medium PGS,Low Clinical Risk
5,European,-0.806482,64.201009,2.228350,0.438690,0.978404,0.603776,45.18,0,NaN,12.620185,Medium PGS,Int Clinical Risk


## Select columns and convert time

In [3]:
feature_cols = [
    'Ancestry',
    'Age',
    'M_Spike',
    'sFLC_Ratio',
    'Creatinine',
    'PGS_Score',
    'PGS_Bin',
]
duration_col = 'Time_Months'
event_col = 'Status'

df = df_raw[feature_cols + [duration_col, event_col]].copy()

# Convert time from months to years
df['Time_Years'] = df['Time_Months'] / 12
df = df.drop(columns=['Time_Months'])

## Rename to lower_snake_case

In [4]:
rename_map = {
    'Ancestry':   'ancestry',
    'Age':        'age',
    'M_Spike':    'm_spike',
    'sFLC_Ratio': 'sflc_ratio',
    'Creatinine': 'creatinine',
    'PGS_Score':  'pgs_score',
    'PGS_Bin':    'pgs_bin',
    'Time_Years': 'time_years',
    'Status':     'status',
}
df = df.rename(columns=rename_map)
df.head()

,ancestry,age,m_spike,sflc_ratio,creatinine,pgs_score,pgs_bin,status,time_years
ID,,,,,,,,,
1,European,69.111929,3.282018,3.010026,0.522791,-0.006056,Medium PGS,1,1.335000
2,European,74.327334,3.468359,3.372553,1.028067,0.996723,Medium PGS,1,0.066667
3,European,67.378696,0.588324,2.718493,0.775595,0.723133,Medium PGS,0,8.948333
4,European,72.769428,1.222824,6.164781,0.943936,0.821519,Medium PGS,1,4.296667
5,European,64.201009,2.228350,0.438690,0.978404,-0.806482,Medium PGS,0,3.765000


## Map ancestry (European → 1, African American → 0)

In [5]:
df['ancestry'] = df['ancestry'].map({'European': 1, 'African American': 0})
print('Ancestry value counts:')
print(df['ancestry'].value_counts())

Ancestry value counts:
ancestry
1    1593
0     407
Name: count, dtype: int64


## Encode PGS_Bin (ordinal: Low=0, Medium=1, High=2)

In [6]:
pgs_bin_map = {'Low PGS': 0, 'Medium PGS': 1, 'High PGS': 2}
df['pgs_bin'] = df['pgs_bin'].map(pgs_bin_map)
print('PGS bin value counts:')
print(df['pgs_bin'].value_counts().sort_index())

PGS bin value counts:
pgs_bin
0     336
1    1373
2     291
Name: count, dtype: int64


## Standardize continuous feature columns

Ancestry is binary (0/1) and is left as-is. All other feature columns are z-score standardized.

In [7]:
continuous_features = ['age', 'm_spike', 'sflc_ratio', 'creatinine', 'pgs_score']
# pgs_bin is ordinal (0/1/2) and is intentionally left on its original scale

scaler = StandardScaler()
df[continuous_features] = scaler.fit_transform(df[continuous_features])

print('Feature statistics after standardization:')
df[continuous_features].describe().round(4)

Feature statistics after standardization:


,age,m_spike,sflc_ratio,creatinine,pgs_score
count,2000.0000,2000.0000,2000.0000,2000.0000,2000.0000
mean,0.0000,-0.0000,0.0000,0.0000,0.0000
std,1.0003,1.0003,1.0003,1.0003,1.0003
min,-2.7340,-1.3972,-0.6103,-2.2628,-3.0406
25%,-0.6713,-0.7451,-0.4775,-0.7038,-0.6924
50%,-0.0010,-0.2383,-0.3108,-0.1120,0.0158
75%,0.6635,0.4854,0.0644,0.6110,0.6760
max,3.7924,6.0289,17.4869,4.2248,3.8405


## Inspect cleaned dataset

In [8]:
print(f'Shape: {df.shape}')
print(f'Missing values:\n{df.isnull().sum()}')
df.head()

Shape: (2000, 9)
Missing values:
ancestry      0
age           0
m_spike       0
sflc_ratio    0
creatinine    0
pgs_score     0
pgs_bin       0
status        0
time_years    0
dtype: int64


,ancestry,age,m_spike,sflc_ratio,creatinine,pgs_score,pgs_bin,status,time_years
ID,,,,,,,,,
1,1,0.137747,2.422286,-0.287442,-1.718453,0.012128,1,1,1.335000
2,1,0.652230,2.640049,-0.247821,0.424321,1.002874,1,1,0.066667
3,1,-0.033230,-0.725654,-0.319304,-0.646363,0.732567,1,0,8.948333
4,1,0.498547,0.015844,0.057345,0.067536,0.829773,1,1,4.296667
5,1,-0.346698,1.190935,-0.568467,0.213709,-0.778696,1,0,3.765000


## Export to cleaned_data/

In [11]:
output_path = OUTPUT_DIR / 'mm_pgs_score.csv'
df.drop(columns=['pgs_bin']).to_csv(output_path)
print(f'Saved {len(df)} rows to {output_path}')

output_path = OUTPUT_DIR / 'mm_nopgs.csv'
df.drop(columns=['pgs_score', 'pgs_bin']).to_csv(output_path)
print(f'Saved {len(df)} rows to {output_path}')

output_path = OUTPUT_DIR / 'mm_pgs_bin.csv'
df.drop(columns=['pgs_score']).to_csv(output_path)
print(f'Saved {len(df)} rows to {output_path}')

Saved 2000 rows to cleaned_data\mm_pgs_score.csv
Saved 2000 rows to cleaned_data\mm_nopgs.csv
Saved 2000 rows to cleaned_data\mm_pgs_bin.csv
